# 1. Initialize Accelerator

In [ ]:
import config
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration
from accelerate.utils import set_seed

set_seed(seed=42)
#using ProjectConfiguration to automatically manage checkpoints
config_project = ProjectConfiguration(
    project_dir=str(config.checkpoint_dir),
    automatic_checkpoint_naming=True,
    total_limit=10
)

accelerator=Accelerator(
    project_config=config_project,
    gradient_accumulation_steps=config.PretrainConfig.accumulate_grad_batches,
)

# 2. Prepare Model,Dataloader,Optimizer and Scheduler

In [ ]:
import torch
from dotenv import load_dotenv
from data import load_data
from modules import GPT
from transformers import get_cosine_schedule_with_warmup
from accelerate.utils.deepspeed import DummyOptim, DummyScheduler

load_dotenv()

loading_ratio=0.01

# only use 1% BookCorpus dataset
with accelerator.main_process_first():
    tokenizer, dataloader = load_data("bookcorpus", loading_ratio=loading_ratio, num_proc=5)

model = GPT(
    vocab_size=config.vocab_size,
    max_len=config.max_len,
    hidden_size=config.hidden_size,
    num_attention_heads=config.num_attention_heads,
    num_hidden_layers=config.num_hidden_layers,
    dropout=config.dropout,
)

if accelerator.state.deepspeed_plugin is not None:
    ds_config = accelerator.state.deepspeed_plugin.deepspeed_config
else:
    ds_config = None
    
num_processes=accelerator.num_processes

# Creates Dummy Optimizer if `optimizer` was specified in the config file else creates Adam Optimizer
if ds_config is not None and "optimizer" in ds_config:
    optimizer = DummyOptim(model.parameters(), 
        lr=config.PretrainConfig.lr,
        weight_decay=config.weight_decay    
    )
else:
    optimizer = model.configure_optimizers(
        lr=config.PretrainConfig.lr,
        weight_decay=config.weight_decay,
        device_type=accelerator.device.type,
    )

# compute correct total steps after dataloader is divided
total_steps = (
    len(dataloader)//num_processes* config.PretrainConfig.n_epoch // accelerator.gradient_accumulation_steps
)
# scale warmup steps
warmup_steps = int(
    config.PretrainConfig.warmup_steps
    * loading_ratio
    // accelerator.gradient_accumulation_steps
)

# Creates Dummy Scheduler if `scheduler` was specified in the config file else creates `args.lr_scheduler_type` Scheduler
if ds_config is not None and "scheduler" in ds_config:
    scheduler = DummyScheduler(
        optimizer,
        total_num_steps=total_steps,
        warmup_num_steps=warmup_steps,
    )
else:
    scheduler = get_cosine_schedule_with_warmup(
        optimizer=optimizer,
        num_training_steps=total_steps,
        num_warmup_steps=warmup_steps,
    )

model, optimizer, scheduler,dataloader = accelerator.prepare(
    model, optimizer, scheduler,dataloader
)

# 3. Define Trainer Class

In [ ]:
import os
import torch.nn as nn
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

class TrainingStatus:
    def __init__(self, checkpoint_interval):
        self.global_step = 0
        self.checkpoint_interval = checkpoint_interval

    def state_dict(self):
        return {
            "global_step": self.global_step,
            "checkpoint_interval": self.checkpoint_interval,
        }

    def load_state_dict(self, state_dict):
        self.global_step = state_dict["global_step"]
        self.checkpoint_interval = state_dict["checkpoint_interval"]

class GPTTrainer:
    def __init__(self, model, tokenizer,dataloader,optimizer,scheduler, accelerator, use_tensorboard=False):
        self.model = model
        self.tokenizer = tokenizer
        self.dataloader=dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.accelerator = accelerator
        self.status = TrainingStatus(config.checkpoint_interval)
        accelerator.register_for_checkpointing(self.status)
        self.criterion = nn.CrossEntropyLoss()

        self.writer = None
        if accelerator.is_main_process and use_tensorboard:
            self.writer = SummaryWriter(log_dir=config.checkpoint_dir / "logs")

    def split_batch(self, batch):
        src, tgt = batch[:, :-1], batch[:, 1:]
        return src, tgt

    def train(self, epoch ,dataloader):
        model = self.model
        tokenizer = self.tokenizer
        dataloader=dataloader
        optimizer = self.optimizer
        scheduler = self.scheduler
        accelerator = self.accelerator
        status = self.status
        writer = self.writer
        criterion=self.criterion

        model.train()
        total_loss = 0
        optimizer.zero_grad()

        process_bar=tqdm(
            dataloader,
            desc=f"Training Epoch {epoch}",
            disable=not accelerator.is_main_process,
        )

        for batch in process_bar:
            with accelerator.accumulate(model):
                src, tgt = self.split_batch(batch)

                # [batch_size, max_len - 1, vocab_size]
                outputs = model(src)

                # [batch_size * (max_len - 1), vocab_size]
                outputs = outputs.contiguous().view(-1, tokenizer.get_vocab_size())
                loss = criterion(outputs, tgt.contiguous().view(-1))

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(),max_norm=config.clip)
                    status.global_step += 1

                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

                total_loss += loss.item()

                if writer is not None:
                    writer.add_scalar("Loss/train", loss.item(), status.global_step)
                    writer.add_scalar("Learning_rate",scheduler.get_last_lr()[0],status.global_step)

        return total_loss,len(dataloader)

    #pass the restore_iteration
    def training_loop(self,restore_iteration=-1):
        accelerator = self.accelerator
        dataloader = self.dataloader
        writer=self.writer
        status=self.status

        restore_path = config.checkpoint_dir/f"checkpoints/checkpoint_{restore_iteration}"
        if restore_iteration != -1 and os.path.exists(restore_path):
            accelerator.load_state(restore_path)
            total_batches=status.global_step*accelerator.gradient_accumulation_steps
            restore_epoch=total_batches//len(dataloader)
            skip_batches=total_batches%len(dataloader)
            skipped_dataloader=accelerator.skip_first_batches(dataloader,skip_batches)
        else:
            restore_epoch = 0
            skipped_dataloader=dataloader

        for epoch in range(restore_epoch, config.PretrainConfig.n_epoch):
            if epoch == restore_epoch:
                current_dataloader = skipped_dataloader
            else:
                current_dataloader = dataloader
            local_loss,local_batch_size = self.train(epoch + 1,current_dataloader)

            if (epoch+1) % config.checkpoint_interval == 0:
                accelerator.save_state()

            gathered_loss = accelerator.gather(torch.tensor(local_loss))
            total_samples = accelerator.gather(torch.tensor(local_batch_size))

            if writer and accelerator.is_main_process:
                avg_train_loss = gathered_loss.sum() / total_samples.sum()
                writer.add_scalar('Loss/train_epoch', avg_train_loss, epoch)

    def save_model(self):
        accelerator = self.accelerator
        model = self.model

        accelerator.wait_for_everyone()
        #save the model
        if accelerator.is_main_process:
            save_path=config.save_model_dir/f"gpt_pretrained.pth"

            unwrapped_model=accelerator.unwrap_model(model)
            accelerator.save(
                unwrapped_model.state_dict(),
                save_path
            )

# 4. Main Program

In [ ]:
trainer=GPTTrainer(model, tokenizer, dataloader, optimizer, scheduler, accelerator, use_tensorboard=True)

trainer.training_loop()
#trainer.save_model()